# Citation Index API — Usage Guide

This notebook demonstrates how to use the **Citation Index API** to extract and parse academic references from PDF documents.

The API exposes three main pipelines, all backed by an asynchronous job queue:

| Pipeline | Endpoint | Input | Output |
|----------|----------|-------|--------|
| **Text Extraction** | `POST /extract/text` | PDF file | Markdown text |
| **Reference Extraction** | `POST /extract/references` | Markdown text | List of raw reference strings |
| **Reference Parsing** | `POST /parse/references` | List of reference strings | Structured bibliographic records |

Every endpoint returns a `job_id` immediately. You then poll for status and retrieve the result once the job completes.

## 0. Setup

In [ ]:
import time
import json
import requests
from pathlib import Path
from pprint import pprint

In [ ]:
# Point this at your running Citation Index API instance
API_BASE = "https://citation-index-api-graphia-app1-staging.apps.bst2.paas.psnc.pl"

# Polling settings
POLL_INTERVAL = 3   # seconds between status checks
MAX_WAIT     = 600  # maximum seconds to wait for a job

## 1. Health Check

Verify the API is reachable and its backing services (Redis, storage) are healthy.

In [ ]:
resp = requests.get(f"{API_BASE}/health")
resp.raise_for_status()
pprint(resp.json())

---

## 2. Helper: Poll a Job Until Completion

All three pipelines are **asynchronous**. The helper below submits a polling loop that:

1. Calls `GET /jobs/{job_id}/status` every few seconds.
2. Returns the final result from `GET /jobs/{job_id}` once the status is `completed`.
3. Raises on failure or timeout.

In [ ]:
def wait_for_job(job_id: str, poll_interval: int = POLL_INTERVAL, max_wait: int = MAX_WAIT) -> dict:
    """Poll a job until it reaches a terminal state and return the result."""
    start = time.time()
    last_status = None

    while (time.time() - start) < max_wait:
        status_resp = requests.get(f"{API_BASE}/jobs/{job_id}/status")
        status_resp.raise_for_status()
        info = status_resp.json()
        status = info["status"]

        if status != last_status:
            elapsed = time.time() - start
            print(f"  [{elapsed:5.1f}s] status = {status}")
            last_status = status

        if status == "completed":
            result_resp = requests.get(f"{API_BASE}/jobs/{job_id}")
            result_resp.raise_for_status()
            return result_resp.json()

        if status == "failed":
            raise RuntimeError(f"Job {job_id} failed: {info.get('error', 'unknown')}")

        time.sleep(poll_interval)

    raise TimeoutError(f"Job {job_id} did not complete within {max_wait}s")

---

## 3. Text Extraction (PDF → Markdown)

Upload a PDF and get back markdown-formatted text.

**Endpoint:** `POST /extract/text`  
**Query params:**
- `extractor` — `"pymupdf"` (default) or `"marker"`
- `markdown` — `true` (default) / `false`

**Body:** multipart file upload (`file` field, `application/pdf`)

In [ ]:
# --- Replace with the path to your own PDF ---
pdf_path = Path("../benchmarks/cex/all_pdfs/COM-SCI_25.pdf")

with open(pdf_path, "rb") as f:
    resp = requests.post(
        f"{API_BASE}/extract/text",
        files={"file": (pdf_path.name, f, "application/pdf")},
        params={"extractor": "pymupdf", "markdown": True},
    )
resp.raise_for_status()

job = resp.json()
print(f"Job submitted: {job['job_id']}  (status: {job['status']})")

In [ ]:
text_result = wait_for_job(job["job_id"])

extracted_text = text_result["text"]
print(f"Extracted {len(extracted_text)} characters.")
print("\n--- First 500 characters ---")
print(extracted_text[:500])

---

## 4. Reference Extraction (Text → Reference Strings)

Send markdown text and get back a list of raw bibliographic reference strings identified by the LLM.

**Endpoint:** `POST /extract/references`  
**Query params:**
- `method` — `"full_text"` (default)
- `temperature` — float, e.g. `0.3`
- `prompt_name` — custom prompt template path (optional)

**Body (JSON):** `{"text": "<markdown text>"}`

In [ ]:
# You can use the text extracted in section 3, or supply your own.
sample_text = """
## Literatur

Aron, Raymond/Dominique Schnapper (1988): Power, modernity, and sociology:
selected sociological writings. Aldershot, Hants, England

Collins, Harry (2004): Gravity's shadow: the search for gravitational waves.
Chicago: University of Chicago Press.

Collins, Harry M. (1981): Stages in the Empirical Programme of Relativism.
In: Social Studies of Science, 11 S. 3-10.

Dosi, Giovanni (1982): Technological Paradigms and Technological Trajectories.
In: Research Policy, 11 S. 147-162.

Kuhn, Thomas S. (1976): Die Struktur wissenschaftlicher Revolutionen.
Frankfurt/Main: Suhrkamp.
"""

resp = requests.post(
    f"{API_BASE}/extract/references",
    json={"text": sample_text},
    params={"method": "full_text", "temperature": 0.3},
)
resp.raise_for_status()

job = resp.json()
print(f"Job submitted: {job['job_id']}  (status: {job['status']})")

In [ ]:
extraction_result = wait_for_job(job["job_id"])

references = extraction_result.get("references", [])
print(f"Found {len(references)} references:\n")
for i, ref in enumerate(references, 1):
    print(f"  {i}. {ref}")

---

## 5. Reference Parsing (Strings → Structured Records)

Send a list of raw reference strings and receive structured bibliographic fields (author, title, year, journal, etc.).

**Endpoint:** `POST /parse/references`  
**Query params:**
- `parser` — `"llm"` (default) or `"grobid"`
- `temperature` — float, e.g. `0.0`
- `prompt_name` — custom prompt template path (optional)

**Body (JSON):** `{"references": ["ref string 1", "ref string 2", ...]}`

In [ ]:
# You can feed in the references from section 4, or supply your own list.
raw_references = [
    "B Algers, G Bertoni, D Broom, J Hartung, L Lidfors, J Metz, L Munksgaard, "
    "T N Pina, P Oltenacu, J Rehage, J Rushen. Scientific report on the effects "
    "of farming systems on dairy cow welfare and disease. Annex to the EFSA Journal. "
    "2009. Vol. 1143",
    "E Burow, T Rousing, P Thomsen, D Otten, J Sørensen. Effect of grazing on "
    "the cow welfare of dairy herds evaluated by a multidimensional welfare index. "
    "Animal. 2013a. Vol. 7",
    "D Gieseke, C Lambertz, M Gauly. Relationship between herd size and animal "
    "welfare in dairy cattle. Journal of Dairy Science. 2018. Vol. 101",
]

resp = requests.post(
    f"{API_BASE}/parse/references",
    json={"references": raw_references},
    params={"parser": "llm", "temperature": 0.0},
)
resp.raise_for_status()

job = resp.json()
print(f"Job submitted: {job['job_id']}  (status: {job['status']})")

In [ ]:
parse_result = wait_for_job(job["job_id"])

parsed_refs = parse_result.get("references", [])
print(f"Parsed {len(parsed_refs)} references:\n")
for ref in parsed_refs:
    pprint(ref)
    print()

---

## 6. End-to-End: PDF → Structured References

Chain all three steps together to go from a raw PDF to fully structured bibliographic records.

In [ ]:
pdf_path = Path("../benchmarks/cex/all_pdfs/PSY_97.pdf")  # replace with your PDF

# Step 1 — Extract text from PDF
print("=" * 60)
print("Step 1: Text Extraction (PDF -> Markdown)")
print("=" * 60)

with open(pdf_path, "rb") as f:
    resp = requests.post(
        f"{API_BASE}/extract/text",
        files={"file": (pdf_path.name, f, "application/pdf")},
        params={"extractor": "pymupdf", "markdown": True},
    )
resp.raise_for_status()
job_id = resp.json()["job_id"]
print(f"Job: {job_id}")

text_result = wait_for_job(job_id)
full_text = text_result["text"]
print(f"Extracted {len(full_text)} characters.\n")

In [ ]:
# Step 2 — Extract references from the markdown text
print("=" * 60)
print("Step 2: Reference Extraction (Text -> Reference Strings)")
print("=" * 60)

resp = requests.post(
    f"{API_BASE}/extract/references",
    json={"text": full_text},
    params={"method": "full_text", "temperature": 0.3},
)
resp.raise_for_status()
job_id = resp.json()["job_id"]
print(f"Job: {job_id}")

extraction_result = wait_for_job(job_id)
ref_strings = extraction_result.get("references", [])
print(f"Found {len(ref_strings)} references.\n")
for i, r in enumerate(ref_strings[:5], 1):
    print(f"  {i}. {r[:120]}..." if len(r) > 120 else f"  {i}. {r}")
if len(ref_strings) > 5:
    print(f"  ... and {len(ref_strings) - 5} more")

In [ ]:
# Step 3 — Parse references into structured records
print("=" * 60)
print("Step 3: Reference Parsing (Strings -> Structured Records)")
print("=" * 60)

resp = requests.post(
    f"{API_BASE}/parse/references",
    json={"references": ref_strings},
    params={"parser": "llm", "temperature": 0.0},
)
resp.raise_for_status()
job_id = resp.json()["job_id"]
print(f"Job: {job_id}")

parse_result = wait_for_job(job_id)
structured_refs = parse_result.get("references", [])
print(f"\nParsed {len(structured_refs)} references into structured records.\n")
pprint(structured_refs[0]) if structured_refs else None

---

## 7. Checking Job Status Directly

You can inspect any job at any time using its `job_id`.

In [ ]:
# Replace with a real job_id from a previous run
example_job_id = job_id  # reuse the last one from above

# GET /jobs/{job_id}/status — lightweight status check
status_resp = requests.get(f"{API_BASE}/jobs/{example_job_id}/status")
print("Status endpoint:")
pprint(status_resp.json())

In [ ]:
# GET /jobs/{job_id} — full result (only meaningful when status = completed)
result_resp = requests.get(f"{API_BASE}/jobs/{example_job_id}")
print(f"Result endpoint (HTTP {result_resp.status_code}):")
pprint(result_resp.json())

You can also request XML output for parsing results:

```
GET /jobs/{job_id}?format=xml
```

In [ ]:
xml_resp = requests.get(f"{API_BASE}/jobs/{example_job_id}", params={"format": "xml"})
if xml_resp.status_code == 200:
    print(xml_resp.text[:1000])
else:
    print(f"XML not available (HTTP {xml_resp.status_code}): {xml_resp.text}")

---

## 8. Batch Processing with Concurrent Requests

The queue system supports multiple concurrent jobs. Submit several at once, then poll them in parallel.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

reference_batches = [
    [
        "Arends-Toth, J. / Van de Vijver, F. J. (2003): Multiculturalism and "
        "acculturation: Views of Dutch and Turkish-Dutch. European Journal of "
        "Social Psychology 33(2), S. 249-266.",
        "Berry, J. (1997): Immigration, acculturation and adaption. Applied "
        "Psychology 46(1), S. 5-34.",
    ],
    [
        "Kuhn, Thomas S. (1976): Die Struktur wissenschaftlicher Revolutionen. "
        "Frankfurt/Main: Suhrkamp.",
        "Luhmann, Niklas (1990): Die Wissenschaft der Gesellschaft, "
        "Frankfurt/Main: Suhrkamp.",
    ],
    [
        "Collins, Harry (2004): Gravity's shadow: the search for gravitational "
        "waves. Chicago: University of Chicago Press.",
        "Dosi, Giovanni (1982): Technological Paradigms and Technological "
        "Trajectories. In: Research Policy, 11 S. 147-162.",
    ],
]


def submit_and_wait(refs, batch_num):
    """Submit a parsing job and wait for the result."""
    resp = requests.post(
        f"{API_BASE}/parse/references",
        json={"references": refs},
        params={"parser": "llm", "temperature": 0.0},
    )
    resp.raise_for_status()
    jid = resp.json()["job_id"]
    print(f"  Batch {batch_num}: submitted as {jid}")
    result = wait_for_job(jid)
    return batch_num, result


print(f"Submitting {len(reference_batches)} parsing jobs concurrently...\n")

with ThreadPoolExecutor(max_workers=len(reference_batches)) as pool:
    futures = [
        pool.submit(submit_and_wait, batch, i + 1)
        for i, batch in enumerate(reference_batches)
    ]

    for future in as_completed(futures):
        batch_num, result = future.result()
        n = len(result.get("references", []))
        print(f"\nBatch {batch_num} completed — {n} references parsed")

---

## 9. API Reference (Quick Cheat Sheet)

| Method | Path | Description |
|--------|------|-------------|
| `GET`  | `/` | API info (name, version, links) |
| `GET`  | `/health` | Health check (Redis + storage status) |
| `GET`  | `/jobs/{job_id}/status` | Lightweight job status |
| `GET`  | `/jobs/{job_id}` | Full job result (`?format=xml` for XML) |
| `POST` | `/extract/text` | Upload PDF → markdown text |
| `POST` | `/extract/references` | Markdown text → reference strings |
| `POST` | `/parse/references` | Reference strings → structured records |

Interactive Swagger docs are available at **`{API_BASE}/docs`**.